# 🚀 Master Deployment Orchestrator

**Complete end-to-end deployment automation for Healthcare Data Solutions**

This notebook orchestrates the entire deployment process by running all deployment notebooks in the correct sequence:

1. **Environment Deployer** - Deploy libraries to Fabric environments
2. **Lakehouses & Tables Deployer** - Create lakehouses and deploy Delta tables
3. **Notebook Deployer** - Format and deploy notebooks to workspace
4. **Pipeline Deployer** - Deploy pipeline templates
5. **PowerBI Deployer** - Deploy Power BI semantic models and reports
6. **Update Admin Config** - Update deployment configuration files
7. **Deployment Validator** - Validate all deployed resources

---

## 📋 Prerequisites

> ⚠️ **IMPORTANT:** Attach the `deployment_lakehouse` (containing `hds-build-artifacts`) to this notebook before running.
> Click **Lakehouses → Add → Existing lakehouse** in the left panel and select the lakehouse where `hds-build-artifacts/` folder is present.
> The bootstrap cell will automatically propagate it to all child deployer notebooks.

Before running this notebook:
- ✅ **Attach the lakehouse containing `hds-build-artifacts`** to this notebook (see above)
- ✅ Update configuration in `common_deployment_config.ipynb`
- ✅ Ensure all source artifacts are available in the distribution path
- ✅ Verify workspace permissions (Contributor or higher)

---

## ⚙️ What This Does

1. **Pre-deployment Setup**
   - Loads common configuration from `common_deployment_config`
   - Discovers all lakehouses in the workspace
   - Attaches lakehouses to all deployed notebooks

2. **Sequential Deployment**
   - Runs each deployment notebook in dependency order
   - Reports progress and any errors
   - Provides deployment summary

3. **Post-deployment Validation**
   - Validates all deployed artifacts
   - Verifies workspace configuration

---

> **💡 Tip:** You can run this entire notebook to perform a complete deployment, or run individual cells to execute specific deployment stages.

## 1️⃣ Imports


In [ ]:
# All imports for master deployer (bootstrap + orchestration)
import json
import base64
import time
from datetime import datetime
from sempy.fabric import FabricRestClient

print("✓ All imports loaded")

## 2️⃣ Bootstrap: Attach Deployment Lakehouse to Child Notebooks

Programmatically attaches `deployment_lakehouse` to the child deployer notebooks so they
have lakehouse context when executed via `%run`. This eliminates the need to manually
attach the lakehouse to each notebook in the Fabric UI.

> **Idempotent** — safe to re-run; overwrites the same dependency blob.

In [ ]:
# ---------------------------------------------------------------------------
# Bootstrap: Attach this notebook's lakehouse to child deployer notebooks
# ---------------------------------------------------------------------------

DEPLOYER_NOTEBOOKS_TO_ATTACH = [
    "common_deployment_config",
    "lakehouses_and_tables_deployer",
    "update_admin_config",
    "powerbi_deployer"
]


def _poll_for_definition(client, location_url, max_wait=120):
    """Poll an async getDefinition operation until status is Succeeded."""
    elapsed = 0
    while elapsed < max_wait:
        resp = client.get(location_url)
        body = json.loads(resp.text)
        status = body.get("status", "")

        if status in ("Succeeded", "succeeded"):
            result_location = resp.headers.get("Location", "")
            if result_location:
                result_resp = client.get(result_location)
                return json.loads(result_resp.text)
            if "definition" in body:
                return body
            result_resp = client.get(f"{location_url}/result")
            return json.loads(result_resp.text)

        if status in ("Failed", "failed"):
            raise RuntimeError(f"getDefinition failed: {body.get('error')}")

        retry_after = int(resp.headers.get("Retry-After", 3))
        time.sleep(retry_after)
        elapsed += retry_after

    raise TimeoutError(f"getDefinition did not complete within {max_wait}s")


def attach_lakehouse_to_notebook(
    client, workspace_id, notebook_name, lakehouse_id, lakehouse_name
):
    """Attach a default lakehouse to a notebook via Fabric API (idempotent)."""
    # 1. Find notebook by display name
    resp = client.get(f"/v1/workspaces/{workspace_id}/notebooks")
    notebooks = json.loads(resp.text).get("value", [])
    nb_match = next(
        (n for n in notebooks if n["displayName"] == notebook_name), None
    )
    if not nb_match:
        print(f"  \u26a0 '{notebook_name}' not found in workspace \u2014 skipping")
        return False

    notebook_id = nb_match["id"]

    # 2. Get current notebook definition in ipynb format (handle async 202)
    resp = client.post(
        f"/v1/workspaces/{workspace_id}/notebooks/{notebook_id}/getDefinition?format=ipynb"
    )
    if resp.status_code == 200:
        body = json.loads(resp.text)
        if "definition" in body:
            defn = body
        else:
            location = resp.headers.get("Location", "")
            if not location:
                print(f"  \u26a0 No definition or Location for '{notebook_name}'")
                return False
            defn = _poll_for_definition(client, location)
    elif resp.status_code == 202:
        location = resp.headers.get("Location", "")
        if not location:
            print(f"  \u26a0 202 but no Location header for '{notebook_name}'")
            return False
        defn = _poll_for_definition(client, location)
    else:
        print(f"  \u26a0 getDefinition returned {resp.status_code} for '{notebook_name}'")
        return False

    parts = defn.get("definition", {}).get("parts", [])
    if not parts:
        print(f"  \u26a0 No parts in definition for '{notebook_name}'")
        return False

    nb_data = json.loads(base64.b64decode(parts[0]["payload"]))

    # 3. Check if already attached (skip update if unchanged)
    existing_lh = (
        nb_data.get("metadata", {})
        .get("dependencies", {})
        .get("lakehouse", {})
    )
    if existing_lh.get("default_lakehouse") == lakehouse_id:
        print(f"  \u2713 {notebook_name} \u2014 already attached (skipped update)")
        return True

    # 4. Inject lakehouse dependency
    meta = nb_data.setdefault("metadata", {})
    deps = meta.setdefault("dependencies", {})
    deps["lakehouse"] = {
        "default_lakehouse": lakehouse_id,
        "default_lakehouse_name": lakehouse_name,
        "default_lakehouse_workspace_id": workspace_id,
    }

    # 5. Update notebook definition
    parts[0]["payload"] = base64.b64encode(
        json.dumps(nb_data).encode()
    ).decode()
    resp = client.post(
        f"/v1/workspaces/{workspace_id}/notebooks/{notebook_id}/updateDefinition?format=ipynb",
        json={"definition": {"format": "ipynb", "parts": parts}},
    )
    ok = resp.status_code in [200, 202]
    status_icon = "\u2713" if ok else "\u2717"
    print(f"  {status_icon} {notebook_name} \u2014 {'attached' if ok else resp.status_code}")
    return ok


# --- Execute bootstrap using current notebook's attached lakehouse ---
ctx = notebookutils.runtime.context
bootstrap_workspace_id = ctx["currentWorkspaceId"]
bootstrap_lakehouse_id = ctx["defaultLakehouseId"]
bootstrap_lakehouse_name = ctx["defaultLakehouseName"]

if not bootstrap_lakehouse_id:
    raise RuntimeError(
        "\u274c No lakehouse attached to master_deployer! "
        "Please attach a lakehouse before running."
    )

bootstrap_client = FabricRestClient()

print(f"\U0001f4ce Attaching '{bootstrap_lakehouse_name}' ({bootstrap_lakehouse_id}) to child notebooks...")
results = []
for nb_name in DEPLOYER_NOTEBOOKS_TO_ATTACH:
    results.append(
        attach_lakehouse_to_notebook(
            bootstrap_client,
            bootstrap_workspace_id,
            nb_name,
            bootstrap_lakehouse_id,
            bootstrap_lakehouse_name,
        )
    )

if not all(results):
    raise RuntimeError("\u274c One or more child notebooks failed lakehouse attachment.")

print("\n\u2713 Bootstrap complete \u2014 child notebooks have lakehouse context.")

## 3️⃣ Build Artifacts Validator

This step runs the `build_artifacts_validator` notebook to validate that required build artifacts are present in the distribution path.

In [ ]:
%run build_artifacts_validator

## 4️⃣ Deployment Execution

### Run All Deployment Notebooks in Sequence

### Step 1: Environment Deployer

Deploy libraries to Fabric environments

In [ ]:
%run environment_deployer

### Step 2: Lakehouses & Tables Deployer

Create lakehouses and deploy Delta tables

In [ ]:
%run lakehouses_and_tables_deployer

### Step 3: Notebook Deployer

Format and deploy notebooks to workspace

In [ ]:
%run notebook_deployer

### Step 4: Pipeline Deployer

Deploy pipeline templates

In [ ]:
%run pipeline_deployer

### Step 5: PowerBI Deployer

Deploy Power BI semantic models and reports

In [ ]:
%run powerbi_deployer

### Step 6: Update Admin Config

Update deployment configuration files

In [ ]:
%run update_admin_config

### Step 7: Deployment Validator

Validate all deployed resources and configurations

## 📊 Deployment Summary

In [ ]:
print("\n" + "=" * 80)
print("📊 DEPLOYMENT SUMMARY")
print("=" * 80)

print("\n✅ Deployment Stages Completed:")
print("  1. ✓ Environment Deployment")
print("  2. ✓ Lakehouses & Tables Deployment")
print("  3. ✓ Notebook Deployment")
print("  4. ✓ Pipeline Deployment")
print("  5. ✓ PowerBI Deployment")
print("  6. ✓ Admin Config Update")
print("  7. ✓ Deployment Validation")

print("\n📋 Configuration:")
print(f"  • Workspace:        {WORKSPACE_NAME}")
print(f"  • Version:          {ARTIFACT_VERSION}")
print(f"  • Solution:         {SOLUTION_NAME}")
print(f"  • Environment:      {TARGET_ENVIRONMENT_NAME or 'Auto-detected'}")

print("\n🗂️  Resources:")
print(f"  • Lakehouses:       {len(LOGICAL_LAKEHOUSES)}")
print(f"  • Notebooks:        {len(NOTEBOOK_LAKEHOUSE_MAPPING)}")
print(f"  • Finished:         {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

print("\n" + "=" * 80)
print("🎉 ALL DEPLOYMENT STAGES COMPLETED SUCCESSFULLY!")
print("=" * 80)

print("\n💡 Next Steps:")
print("  1. Verify deployed artifacts in the workspace")
print("  2. Review lakehouse attachments for all notebooks")
print("  3. Test pipeline execution with sample data")
print("  4. Validate environment library installation")
print("  5. Review Power BI semantic models and reports")
print("  6. Check deployment validation results")
print("\n" + "=" * 80 + "\n")

In [ ]:
%run deployment_validator